# =====================================================================
# NOTEBOOK 2: OPERATIONAL PREDICTION PIPELINE
# =====================================================================
## Step 1: Environment Setup & Architecture Definition
This initialization cell establishes the hardware and directory configurations required for the prediction pipeline. It enforces strict threading limits to prevent CPU thrashing in the WSL Ubuntu environment and maps the output paths directly to the Rclone-mounted Google Drive. Additionally, it loads the `DeepFireCNN` model architecture and the `LazyDiskDataset` loader required to process the spatial tensors.

In [1]:
# =====================================================================
# NOTEBOOK 2: OPERATIONAL PREDICTION WITH T-TEST DATA IMPUTATION
# =====================================================================
import os
import gc
import glob
import numpy as np
import rasterio
from rasterio.mask import mask
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import mapping
import folium
import base64
import branca.colormap as cm
from PIL import Image
import io
from datetime import datetime, timedelta
from scipy import stats
from scipy.ndimage import label
import ee
from tqdm import tqdm

# -----------------------------------------------------------------
# 1. HARDWARE & ENVIRONMENT SETUP
# -----------------------------------------------------------------
os.environ.update({"OMP_NUM_THREADS":"1", "OPENBLAS_NUM_THREADS":"1", "MKL_NUM_THREADS":"1", "VECLIB_MAXIMUM_THREADS":"1", "NUMEXPR_NUM_THREADS":"1"})
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Data"
sliced_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Sliced_Data"
model_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Models"
pred_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Prediction"
os.makedirs(pred_dir, exist_ok=True)

EXPECTED_CHANNELS = (5 * 14) + 5
horizons = [1, 7, 14, 30]

class DeepFireCNN(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.GroupNorm(8, 256), nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.Conv2d(256, 1, 1)
        )
    def forward(self, x): return self.net(x)

class LazyDiskDataset(Dataset):
    def __init__(self, directory, phase, seq_len=14, horizon=7):
        self.files = glob.glob(os.path.join(directory, f"{phase}_patch_*.pt"))
        self.seq_len = seq_len
        self.horizon = horizon
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        data = torch.load(self.files[idx], weights_only=False)
        time_steps = data['time_steps']
        max_start = time_steps - self.seq_len - self.horizon
        t_idx = np.random.randint(0, max_start)
        env_chunks = [data['env'][var_idx, t_idx : t_idx+self.seq_len] for var_idx in range(5)]
        static_chunks = [data['lulc'][t_idx+self.seq_len-1].unsqueeze(0), data['roads'].unsqueeze(0), data['dem'].unsqueeze(0), data['slope'].unsqueeze(0), data['aspect'].unsqueeze(0)]
        inputs = torch.cat(env_chunks + static_chunks, dim=0)
        future_window = data['fire'][t_idx + self.seq_len : t_idx + self.seq_len + self.horizon]
        target = future_window.max(dim=0)[0].unsqueeze(0)
        return inputs, target


## Step 2: Google Earth Engine (GEE) Synchronization
This step verifies the integrity of the local dataset against the required parameters for the operational model. It authenticates with Google Earth Engine (Project ID: _replaced with yoyr project_) and scans the local Drive directory. If any historical or current-year variables (Temperature, VPD, EVI, Soil Moisture, CO, LULC, FireMask, or Topography) are missing, it automatically triggers an asynchronous batch export from GEE at a 1 km spatial resolution.

In [ ]:
# =====================================================================
# RERUN ALWAYS TO SEEK FOR THE UPDATED FILES IN GOOGLE DRIVE AND QUEUE MISSING ONES TO GEE
# =====================================================================
# STEP 2: GOOGLE EARTH ENGINE ASYNCHRONOUS BATCH QUEUEING (1 KM)
# =====================================================================
print(f"\n{'#'*70}\n STEP 2: CHECKING FILES & QUEUEING 1 KM GEE EXPORTS TO DRIVE\n{'#'*70}")
GCP_PROJECT_ID = 'indigo-syntax-420618'

try:
    ee.Initialize(project=GCP_PROJECT_ID)
except Exception:
    ee.Authenticate() 
    ee.Initialize(project=GCP_PROJECT_ID)

from datetime import datetime, date
import time

# Define the precise spatial bounding box for Mato Grosso
mt_state = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', 'Mato Grosso'))
roi_bounds = mt_state.geometry().bounds()
region_coords = roi_bounds.getInfo()['coordinates']

missing_files = 0
found_files = 0
current_year = datetime.now().year

# =====================================================================
# AUTOMATIC FRESHNESS CHECK FOR ONGOING YEAR
# =====================================================================
print(f"--- Enforcing fresh daily data for ongoing year ({current_year}) ---")
today = date.today()
dynamic_prefixes = ['ERA5_Temp', 'ERA5_VPD', 'MODIS_EVI', 'SMAP_SoilMoisture', 'Sentinel5P_CO', 'MODIS_Fire', 'MapBiomas_LULC']

for prefix in dynamic_prefixes:
    f_path = os.path.join(data_dir, f'{prefix}_{current_year}.tif')
    if os.path.exists(f_path):
        file_date = date.fromtimestamp(os.path.getmtime(f_path))
        if file_date < today:
            print(f"  [OUTDATED] {os.path.basename(f_path)} is from {file_date}. Deleting to force GEE update...")
            os.remove(f_path)
            
            # Purge corresponding sliced patches so Step 3 rebuilds them with the new data
            sliced_pattern = os.path.join(sliced_dir, f"*_patch_{current_year}_*.pt")
            for pt_file in glob.glob(sliced_pattern):
                os.remove(pt_file)

# The GEE Asynchronous Export Function - SET TO 1 KM RESOLUTION (1000 meters)
def queue_export(image, description, file_prefix, scale=1000):
    ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder='Data', 
        fileNamePrefix=file_prefix,
        region=region_coords,
        scale=scale,
        crs='EPSG:4326',
        maxPixels=1e13
    ).start()

# =====================================================================
# 1. STATIC VARIABLES (Exported once, year-independent)
# =====================================================================
print("\n--- Checking Static Topography & Infrastructure ---")

f_roads = os.path.join(data_dir, 'Infrastructure_Roads.tif')
if not os.path.exists(f_roads):
    print("  [MISSING] Queueing Infrastructure Roads export...")
    roads = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Central-South-America").filterBounds(roi_bounds)
    distance_to_roads = roads.distance(searchRadius=50000, maxError=100)
    queue_export(distance_to_roads, 'Infrastructure_Roads', 'Infrastructure_Roads')
    missing_files += 1
else:
    found_files += 1

f_dem = os.path.join(data_dir, 'Terrain_DEM.tif')
f_slope = os.path.join(data_dir, 'Terrain_Slope.tif')
f_aspect = os.path.join(data_dir, 'Terrain_Aspect.tif')
srtm = ee.Image("USGS/SRTMGL1_003")

if not os.path.exists(f_dem):
    print("  [MISSING] Queueing DEM export...")
    queue_export(srtm.select('elevation'), 'Terrain_DEM', 'Terrain_DEM')
    missing_files += 1
else: found_files += 1

if not os.path.exists(f_slope):
    print("  [MISSING] Queueing Slope export...")
    queue_export(ee.Terrain.slope(srtm), 'Terrain_Slope', 'Terrain_Slope')
    missing_files += 1
else: found_files += 1

if not os.path.exists(f_aspect):
    print("  [MISSING] Queueing Aspect export...")
    queue_export(ee.Terrain.aspect(srtm), 'Terrain_Aspect', 'Terrain_Aspect')
    missing_files += 1
else: found_files += 1

# =====================================================================
# 2. DYNAMIC VARIABLES (Exported per year)
# =====================================================================
# Dynamically set the range up to the current year
years = range(2018, current_year + 1)

for year in years:
    print(f"\n--- Checking Year {year} ---")
    start_date = f'{year}-01-01'
    end_date   = f'{year+1}-01-01'

    f_temp = os.path.join(data_dir, f'ERA5_Temp_{year}.tif')
    if not os.path.exists(f_temp):
        print(f"  [MISSING] Queueing Temperature for {year}...")
        era5_t = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterBounds(roi_bounds).filterDate(start_date, end_date)
        queue_export(era5_t.select('temperature_2m').toBands(), f'ERA5_Temp_{year}', f'ERA5_Temp_{year}')
        missing_files += 1
    else: found_files += 1

    f_vpd = os.path.join(data_dir, f'ERA5_VPD_{year}.tif')
    if not os.path.exists(f_vpd):
        print(f"  [MISSING] Queueing VPD for {year}...")
        def compute_daily_vpd(image):
            t_k = image.select('temperature_2m')
            td_k = image.select('dewpoint_temperature_2m')
            t_c = t_k.subtract(273.15)
            td_c = td_k.subtract(273.15)
            es = t_c.divide(t_c.add(237.3)).multiply(17.27).exp().multiply(0.6108)
            ea = td_c.divide(td_c.add(237.3)).multiply(17.27).exp().multiply(0.6108)
            vpd = es.subtract(ea).max(ee.Image.constant(0)).rename('VPD')
            return vpd.copyProperties(image, ['system:time_start'])

        vpd_col = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                     .filterBounds(roi_bounds)
                     .filterDate(start_date, end_date)
                     .select(['temperature_2m', 'dewpoint_temperature_2m'])
                     .map(compute_daily_vpd))
        queue_export(vpd_col.toBands(), f'ERA5_VPD_{year}', f'ERA5_VPD_{year}')
        missing_files += 1
    else: found_files += 1

    f_evi = os.path.join(data_dir, f'MODIS_EVI_{year}.tif')
    if not os.path.exists(f_evi):
        print(f"  [MISSING] Queueing MODIS EVI for {year}...")
        evi_col = ee.ImageCollection("MODIS/061/MOD13A2").filterBounds(roi_bounds).filterDate(start_date, end_date).select('EVI')
        queue_export(evi_col.toBands(), f'MODIS_EVI_{year}', f'MODIS_EVI_{year}')
        missing_files += 1
    else: found_files += 1

    f_smap = os.path.join(data_dir, f'SMAP_SoilMoisture_{year}.tif')
    if not os.path.exists(f_smap):
        print(f"  [MISSING] Queueing SMAP Soil Moisture for {year}...")
        def fill_smap_gap(day_offset):
            d = ee.Date(start_date).advance(ee.Number(day_offset), 'day')
            daily = ee.ImageCollection("NASA/SMAP/SPL3SMP_E/006").filterBounds(roi_bounds).filterDate(d, d.advance(1, 'day')).select('soil_moisture_am')
            filled = ee.Image(ee.Algorithms.If(daily.size().gt(0), daily.mean(), ee.Image.constant(-9999))).rename('soil_moisture_am').toFloat()
            return filled.set('system:time_start', d.millis())

        start_ee = ee.Date(start_date)
        end_ee = ee.Date(end_date)
        n_days = end_ee.difference(start_ee, 'day')
        days_seq = ee.List.sequence(0, n_days.subtract(1))
        smap_col = ee.ImageCollection.fromImages(days_seq.map(fill_smap_gap))
        queue_export(smap_col.toBands(), f'SMAP_SoilMoisture_{year}', f'SMAP_SoilMoisture_{year}')
        missing_files += 1
    else: found_files += 1

    f_co = os.path.join(data_dir, f'Sentinel5P_CO_{year}.tif')
    if not os.path.exists(f_co):
        print(f"  [MISSING] Queueing Sentinel-5P CO for {year}...")
        def get_daily_co(day_offset):
            d = ee.Date(start_date).advance(ee.Number(day_offset), 'day')
            daily_col = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CO").filterBounds(roi_bounds).filterDate(d, d.advance(1, 'day')).select('CO_column_number_density')
            daily_img = ee.Image(ee.Algorithms.If(daily_col.size().gt(0), daily_col.mean(), ee.Image.constant(0))).rename('CO').toFloat()
            return daily_img.set('system:time_start', d.millis())

        co_col = ee.ImageCollection.fromImages(days_seq.map(get_daily_co))
        queue_export(co_col.toBands(), f'Sentinel5P_CO_{year}', f'Sentinel5P_CO_{year}')
        missing_files += 1
    else: found_files += 1

    f_fire = os.path.join(data_dir, f'MODIS_Fire_{year}.tif')
    if not os.path.exists(f_fire):
        print(f"  [MISSING] Queueing Fire Data for {year}...")
        col = ee.ImageCollection("MODIS/061/MOD14A1").filterBounds(roi_bounds).filterDate(start_date, end_date).select('FireMask')
        queue_export(col.toBands(), f'MODIS_Fire_{year}', f'MODIS_Fire_{year}')
        missing_files += 1
    else: found_files += 1

    f_lulc = os.path.join(data_dir, f'MapBiomas_LULC_{year}.tif')
    if not os.path.exists(f_lulc):
        print(f"  [MISSING] Queueing Land Cover for {year}...")
        mb_year = min(year, 2023)
        mb_band = f'classification_{mb_year}'
        mapbiomas = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1").select(mb_band).clip(mt_state)
        queue_export(mapbiomas, f'MapBiomas_LULC_{year}', f'MapBiomas_LULC_{year}')
        missing_files += 1
    else: found_files += 1

print("\n" + "=" * 60)
print("1 KM RESOLUTION INVENTORY (Operational Model):")
print("  Static  : Roads Distance, DEM, Slope, Aspect")
print("  Dynamic : Temperature, VPD, MODIS EVI, SMAP Soil Moisture,")
print("            MODIS Fire (target), MapBiomas LULC, Sentinel-5P CO")
print("-" * 60)
if missing_files == 0:
    print(f"[SUCCESS] ALL {found_files} REQUIRED FILES FOUND LOCALLY!")
    print("You are ready to proceed to Step 3 (Spatial Tiling).")
else:
    print(f"[STATUS] Found {found_files} files locally.")
    print(f"[ACTION] Queued {missing_files} missing files to Earth Engine at 1km Scale.")
    print("Monitor the GEE Task Manager at https://code.earthengine.google.com/tasks")
    print("Wait for Drive sync to complete before running Step 3.")
print("=" * 60 + "\n")

## Step 3: Spatial Tiling & Tensor Normalization
To prepare the raw `.tif` files for the neural network, this cell executes a spatial alignment and tiling process. It dynamically truncates temporal dimensions to match the latest available data, upsamples 16-day EVI composites to daily sequences, and normalizes the continuous variables to prevent 16-bit overflow during inference. The finalized 128x128 matrices are saved as PyTorch `.pt` patches.

In [2]:
# =====================================================================
# RERUN ALWAYS TO SEEK FOR THE SPATIAL TILLED FILES IN GOOGLE DRIVE AND PROCEED TO SLICE THEM LOCALLY
# =====================================================================
# STEP 3: SPATIAL TILING ALIGNMENT (128x128)
# =====================================================================
print(f"\n{'#'*70}\n STEP 3: SPATIAL TILING ALIGNMENT (128x128)\n{'#'*70}")

patch_size = 128
stride = 128
years = range(2018, 2027)

for y in years:
    check_file = os.path.join(sliced_dir, f"{'val' if y >= 2022 else 'train'}_patch_{y}_0.pt")
    requires_slicing = not os.path.exists(check_file)

    if requires_slicing:
        print(f"    -> Loading {y} spatial matrices into RAM (Excluding Precip)...")
        with rasterio.open(os.path.join(data_dir, f'ERA5_Temp_{y}.tif')) as src: t_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, f'ERA5_VPD_{y}.tif')) as src: v_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, f'SMAP_SoilMoisture_{y}.tif')) as src: sm_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, f'Sentinel5P_CO_{y}.tif')) as src: co_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, f'MODIS_EVI_{y}.tif')) as src: evi_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, f'MODIS_Fire_{y}.tif')) as src: f_raw = np.isin(np.nan_to_num(src.read()), [7,8,9]).astype(np.float16)
        
        with rasterio.open(os.path.join(data_dir, f'MapBiomas_LULC_{y}.tif')) as src: l_raw = np.nan_to_num(src.read()).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, 'Infrastructure_Roads.tif')) as src: roads_static = np.nan_to_num(src.read()[0]).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, 'Terrain_DEM.tif')) as src: dem_static = np.nan_to_num(src.read()[0]).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, 'Terrain_Slope.tif')) as src: slope_static = np.nan_to_num(src.read()[0]).astype(np.float16)
        with rasterio.open(os.path.join(data_dir, 'Terrain_Aspect.tif')) as src: aspect_static = np.nan_to_num(src.read()[0]).astype(np.float16)

        # 1. TEMPORAL FIX: Find the minimum time WITHOUT EVI (EVI is a 16-day composite, usually 23 bands)
        min_t = min(t_raw.shape[0], v_raw.shape[0], sm_raw.shape[0], co_raw.shape[0], f_raw.shape[0])
        
        min_h = min(t_raw.shape[1], v_raw.shape[1], sm_raw.shape[1], co_raw.shape[1], evi_raw.shape[1], f_raw.shape[1], roads_static.shape[0], dem_static.shape[0])
        min_w = min(t_raw.shape[2], v_raw.shape[2], sm_raw.shape[2], co_raw.shape[2], evi_raw.shape[2], f_raw.shape[2], roads_static.shape[1], dem_static.shape[1])
        
        print(f"       -> Truncating daily dimensions to {min_t} days and {min_h}x{min_w} spatial grid.")

        # 2. INTERPOLATION: Stretch the 23 EVI bands to cover the full daily sequence (e.g., 365 days)
        if evi_raw.shape[0] < min_t:
            print(f"       -> Upsampling MODIS EVI from {evi_raw.shape[0]} composites to {min_t} daily bands...")
            evi_indices = np.linspace(0, evi_raw.shape[0] - 1, min_t).astype(int)
            evi_raw = evi_raw[evi_indices]

        # Now slice safely
        t = t_raw[:min_t, :min_h, :min_w]
        v = v_raw[:min_t, :min_h, :min_w]
        sm = sm_raw[:min_t, :min_h, :min_w]
        co = co_raw[:min_t, :min_h, :min_w]
        evi = evi_raw[:min_t, :min_h, :min_w]
        f = f_raw[:min_t, :min_h, :min_w]
        
        if l_raw.ndim == 3:
            l = np.repeat(l_raw[:, :min_h, :min_w], min_t, axis=0)[:min_t]
        else:
            l = np.repeat(l_raw[np.newaxis, :min_h, :min_w], min_t, axis=0)[:min_t]

        roads_static = roads_static[:min_h, :min_w]
        dem_static = dem_static[:min_h, :min_w]
        slope_static = slope_static[:min_h, :min_w]
        aspect_static = aspect_static[:min_h, :min_w]

        del t_raw, v_raw, sm_raw, co_raw, evi_raw, f_raw, l_raw
        
        # [CORRECTED SECTION] Float32 casting for statistical reduction to prevent 16-bit Overflow
        for arr in [t, v, sm, co, evi]:
            mean_val = np.mean(arr, dtype=np.float32)
            std_val = np.std(arr, dtype=np.float32)
            arr[:] = ((arr.astype(np.float32) - mean_val) / (std_val + 1e-8)).astype(np.float16)

        for static_arr in [roads_static, dem_static, slope_static, aspect_static]:
            mean_val = np.mean(static_arr, dtype=np.float32)
            std_val = np.std(static_arr, dtype=np.float32)
            static_arr[:] = ((static_arr.astype(np.float32) - mean_val) / (std_val + 1e-8)).astype(np.float16)

        time_steps = min_t
        patch_id = 0
        
        print(f"       -> Slicing tensors into {patch_size}x{patch_size} grids and streaming to Drive...")
        for i in tqdm(range(0, t.shape[1] - patch_size + 1, stride), desc=f"Slicing {y}"):
            for j in range(0, t.shape[2] - patch_size + 1, stride):
                if np.sum(f[:, i:i+patch_size, j:j+patch_size]) == 0 and np.random.rand() > 0.15:
                    continue
                
                env_stack = np.stack([
                    t[:, i:i+patch_size, j:j+patch_size], 
                    v[:, i:i+patch_size, j:j+patch_size],
                    sm[:, i:i+patch_size, j:j+patch_size], 
                    co[:, i:i+patch_size, j:j+patch_size],
                    evi[:, i:i+patch_size, j:j+patch_size]
                ])
                
                phase = "val" if y >= 2022 else "train"
                
                torch.save({
                    'env': torch.from_numpy(env_stack).float(),
                    'fire': torch.from_numpy(f[:, i:i+patch_size, j:j+patch_size]).float(),
                    'lulc': torch.from_numpy(l[:, i:i+patch_size, j:j+patch_size]).float(),
                    'roads': torch.from_numpy(roads_static[i:i+patch_size, j:j+patch_size]).float(),
                    'dem': torch.from_numpy(dem_static[i:i+patch_size, j:j+patch_size]).float(),
                    'slope': torch.from_numpy(slope_static[i:i+patch_size, j:j+patch_size]).float(),
                    'aspect': torch.from_numpy(aspect_static[i:i+patch_size, j:j+patch_size]).float(),
                    'time_steps': time_steps
                }, os.path.join(sliced_dir, f"{phase}_patch_{y}_{patch_id}.pt"))
                patch_id += 1
                
        del t, v, sm, co, evi, f, l
        gc.collect()

# (5 dynamic vars * 14 days) + 1 lulc + 1 roads + 1 dem + 1 slope + 1 aspect = 75 channels
EXPECTED_CHANNELS = (5 * 14) + 5
print(f"\n[INFO] Step 3 Complete. Expected CNN Input Channels: {EXPECTED_CHANNELS}")


######################################################################
 STEP 3: SPATIAL TILING ALIGNMENT (128x128)
######################################################################

[INFO] Step 3 Complete. Expected CNN Input Channels: 75


## Step 5: Inference, Imputation & Cartography
This is the core operational block. It executes a two-phase process:
1. **Dynamic Imputation (Step 5A):** Requests a target prediction date (`DD/MM/YYYY`). If the 14-day trailing sequence is incomplete, it performs a per-variable Welch's T-Test to find the most statistically similar historical year to impute missing continuous data, while using a Last Observation Carried Forward (LOCF) approach for the categorical MapBiomas LULC data.
2. **Prediction & Mapping (Step 5B):** Runs the forward pass of the CNN for the 1, 7, 14, and 30-day horizons. It then translates the output probabilities into operational cartography, saving raw `.tif` files, a 1200 DPI static JPG comparison grid, and an interactive HTML Folium map clipped to the Mato Grosso state boundaries.

In [3]:
# =====================================================================
# STEP 5A: DYNAMIC DATE SELECTION & PER-VARIABLE IMPUTATION
# =====================================================================
print(f"\n{'#'*70}\n STEP 5A: DYNAMIC DATE SELECTION & PER-VARIABLE IMPUTATION\n{'#'*70}")

from datetime import datetime
import scipy.stats as stats

VAR_NAMES = ["Temperature", "VPD", "Soil Moisture", "CO", "EVI"]

# Updated input prompt and parsing to strictly use DD/MM/YYYY
target_date_str = input("\n[INPUT] Enter the target date to predict (DD/MM/YYYY): ")
target_date = datetime.strptime(target_date_str, "%d/%m/%Y")

print(f"[INFO] Target Prediction Date: {target_date.strftime('%Y-%m-%d')}")
target_year = target_date.year
target_day_of_year = target_date.timetuple().tm_yday

def get_latest_patch_for_year(year):
    pattern = os.path.join(sliced_dir, f"*patch_{year}_*.pt")
    files = glob.glob(pattern)
    if not files:
        return None
    return torch.load(max(files, key=os.path.getmtime), weights_only=False)

current_year_data = get_latest_patch_for_year(target_year)
historical_years = [y for y in range(2018, target_year)]

available_days = 0
if current_year_data:
    available_days = min(14, current_year_data['time_steps'] - (target_day_of_year - 14))
    if available_days < 0:
        available_days = 0

print(f"[STATUS] Sequence Check: {available_days}/14 required days available for {target_year}.")

final_env_chunks = []
lulc_chunk = None

if available_days == 14:
    # CONDITION 1: FULL DATA — no imputation needed
    print("[SUCCESS] Full 14-day temporal sequence available. No imputation required.")
    t_idx = target_day_of_year - 14
    final_env_chunks = [current_year_data['env'][var_idx, t_idx : t_idx + 14] for var_idx in range(5)]
    lulc_chunk = current_year_data['lulc'][-1].unsqueeze(0)
    base_static_data = current_year_data

else:
    # CONDITION 2 & 3: PARTIAL OR NO DATA
    print(f"[WARNING] Missing {14 - available_days} days. "
          f"Initiating per-variable Welch T-Test historical matching...")

    t_start = target_day_of_year - 14
    t_end   = t_start + available_days

    # Pre-load all historical patches once to avoid redundant disk reads
    hist_cache = {}
    for hy in historical_years:
        patch = get_latest_patch_for_year(hy)
        if patch is not None:
            hist_cache[hy] = patch

    if not hist_cache:
        raise RuntimeError("[CRITICAL] No historical patches found. Cannot impute.")

    best_year_per_var  = {}   
    best_pval_per_var  = {}   

    for var_idx in range(5):
        print(f"\n  [VAR {var_idx}] {VAR_NAMES[var_idx]}")

        # Build the reference distribution from available current-year days
        if available_days > 0 and current_year_data is not None:
            current_dist = (current_year_data['env'][var_idx, t_start:t_end]
                            .mean(dim=(1, 2)).numpy())
            use_ytd = False
        elif current_year_data is not None and current_year_data['time_steps'] > 0:
            # Window not yet reached: use year-to-date distribution as proxy
            current_dist = (current_year_data['env'][var_idx, :current_year_data['time_steps']]
                            .mean(dim=(1, 2)).numpy())
            use_ytd = True
            print(f"       -> No window data; using YTD distribution ({len(current_dist)} days) as proxy.")
        else:
            # Entirely missing current year: fall back to most recent historical year
            best_year_per_var[var_idx] = max(hist_cache.keys())
            best_pval_per_var[var_idx] = float('nan')
            print(f"       -> No current-year data at all. Falling back to most recent year: "
                  f"{best_year_per_var[var_idx]}")
            continue

        best_year  = None
        best_pval  = -1.0

        for hy, hdata in hist_cache.items():
            if use_ytd:
                hist_dist = (hdata['env'][var_idx, :len(current_dist)]
                             .mean(dim=(1, 2)).numpy())
            else:
                hist_dist = (hdata['env'][var_idx, t_start:t_end]
                             .mean(dim=(1, 2)).numpy())

            if len(current_dist) < 2 or len(hist_dist) < 2:
                p_val = 1.0  
            else:
                _, p_val = stats.ttest_ind(current_dist, hist_dist, equal_var=False)

            if p_val > best_pval:
                best_pval = p_val
                best_year = hy

        best_year_per_var[var_idx] = best_year
        best_pval_per_var[var_idx] = best_pval

    # Summary table
    print(f"\n{'='*62}")
    print(f"  Per-variable imputation sources")
    print(f"  {'Variable':<18} {'Best Year':>10} {'p-value':>10}")
    print(f"  {'-'*42}")
    for var_idx in range(5):
        pv = best_pval_per_var[var_idx]
        pv_str = f"{pv:.4f}" if not (isinstance(pv, float) and np.isnan(pv)) else "fallback"
        print(f"  {VAR_NAMES[var_idx]:<18} {best_year_per_var[var_idx]:>10} {pv_str:>10}")
    print(f"{'='*62}\n")

    # Assemble each continuous channel from its own best-matching historical year
    for var_idx in range(5):
        src_year  = best_year_per_var[var_idx]
        src_data  = hist_cache[src_year]

        if available_days > 0 and current_year_data is not None:
            current_chunk  = current_year_data['env'][var_idx, t_start : t_start + available_days]
            imputed_chunk  = src_data['env'][var_idx, t_start + available_days : t_start + 14]
            merged         = torch.cat([current_chunk, imputed_chunk], dim=0)
            print(f"  [{VAR_NAMES[var_idx]}] {available_days} days current + "
                  f"{14 - available_days} days from {src_year}")
        else:
            t_idx  = target_day_of_year - 14
            merged = src_data['env'][var_idx, t_idx : t_idx + 14]
            print(f"  [{VAR_NAMES[var_idx]}] Fully imputed from {src_year}")

        final_env_chunks.append(merged)

    # -----------------------------------------------------------------
    # CATEGORICAL LULC IMPUTATION (Last Observation Carried Forward)
    # -----------------------------------------------------------------
    print(f"\n  [LULC / Uso do Solo]")
    if current_year_data is not None and current_year_data['time_steps'] > 0:
        lulc_chunk = current_year_data['lulc'][-1].unsqueeze(0)
        print(f"  -> Sourced from current year spatial matrix.")
    else:
        recent_hist_year = max(hist_cache.keys())
        lulc_chunk = hist_cache[recent_hist_year]['lulc'][-1].unsqueeze(0)
        print(f"  -> Categorical data imputed using closest historical year: {recent_hist_year}")

    base_static_data = current_year_data if current_year_data is not None else hist_cache[max(hist_cache.keys())]

# Assemble final input tensor
static_chunks = [
    lulc_chunk,
    base_static_data['roads'].unsqueeze(0),
    base_static_data['dem'].unsqueeze(0),
    base_static_data['slope'].unsqueeze(0),
    base_static_data['aspect'].unsqueeze(0),
]

x_input = torch.cat(final_env_chunks + static_chunks, dim=0).unsqueeze(0).to(device)
print(f"\n[SUCCESS] Final temporal-spatial tensor generated. Shape: {x_input.shape}")

# -----------------------------------------------------------------
# STEP 5B: NEURAL NETWORK INFERENCE & CARTOGRAPHY
# -----------------------------------------------------------------
print(f"\n{'#'*70}\n STEP 5B: EXECUTING CNN PREDICTION & MAPPING\n{'#'*70}")

# Load Shapefiles
path_mun = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data/MT_Municipios_2022/MT_Municipios_2022.shp"
path_fed = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data/Federal Highways/SNV_202410A.shp"
path_est = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data/State Highways/INTERMAT_TRA_SISTEMA_VIARIO_L.shp"

mt_gdf = gpd.read_file(path_mun).to_crs(epsg=4326)
mt_gdf['geometry'] = mt_gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)
mt_gdf = mt_gdf.dropna(subset=['geometry'])
mt_state_boundary = mt_gdf.dissolve()
mt_geom = [mapping(mt_state_boundary.geometry.iloc[0])] 
city_col = 'NM_MUN' if 'NM_MUN' in mt_gdf.columns else mt_gdf.columns[0]
mt_gdf = mt_gdf[[city_col, 'geometry']]

fed_gdf = gpd.clip(gpd.read_file(path_fed).to_crs(epsg=4326), mt_state_boundary)
fed_gdf['geometry'] = fed_gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)
fed_gdf = fed_gdf.dropna(subset=['geometry'])[['geometry']]

state_gdf = gpd.clip(gpd.read_file(path_est).to_crs(epsg=4326), mt_state_boundary)
state_gdf['geometry'] = state_gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)
state_gdf = state_gdf.dropna(subset=['geometry'])[['geometry']]

ref_tif = os.path.join(data_dir, 'ERA5_Temp_2026.tif')
pred_maps = {}

with rasterio.open(ref_tif) as src:
    meta = src.meta.copy()
    meta.update(dtype=rasterio.float32, count=1, nodata=np.nan) 
    map_bounds = [[src.bounds.bottom, src.bounds.left], [src.bounds.top, src.bounds.right]]

for h in horizons:
    model_path = os.path.join(model_dir, f"final_model_{h}d_period.pth")
    if not os.path.exists(model_path): continue
        
    print(f" -> Executing forward pass for {h}-day horizon...")
    inference_model = DeepFireCNN(EXPECTED_CHANNELS).to(device)
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    inference_model.load_state_dict({k.replace('_orig_mod.', ''): v for k, v in state_dict.items()})
    inference_model.eval()
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            raw_logits = inference_model(x_input)
        probabilities = torch.sigmoid(raw_logits).squeeze().cpu().numpy()
        
    # Saves the .tif file for the horizon
    output_tif = os.path.join(pred_dir, f"Fire_Risk_{h}Days_Generated_{target_date.strftime('%Y_%m_%d')}.tif")
    with rasterio.open(output_tif, 'w', **meta) as dst:
        dst.write(probabilities.astype(np.float32), 1)
        
    with rasterio.open(output_tif) as src_clip:
        out_image, out_transform = mask(src_clip, mt_geom, crop=True, filled=False)
        out_meta = src_clip.meta.copy()
        out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform})
        
    with rasterio.open(output_tif, "w", **out_meta) as dest:
        dest.write(out_image)
    
    pred_maps[h] = out_image[0]
    del inference_model, raw_logits, state_dict
    gc.collect()
    torch.cuda.empty_cache()

# Exporting Visualizations
if len(pred_maps) > 0:
    print("\n[INFO] Assembling and saving visualizations...")
    titles_pt = {1: "Amanhã (24h)", 7: "Próximos 7 Dias", 14: "Próximos 14 Dias", 30: "Próximos 30 Dias"}
    cmap_base = mcolors.LinearSegmentedColormap.from_list("BlueYellowRed", ["blue", "yellow", "red"])
    
    print("       -> Generating ultra-high resolution JPG image (1x4 Grid) at 1200 DPI...")
    cmap_jpg = cmap_base.copy()
    cmap_jpg.set_bad(color='white') 
    
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    for idx, h in enumerate(horizons):
        if h in pred_maps:
            ax = axes[idx]
            im = ax.imshow(pred_maps[h], cmap=cmap_jpg, vmin=0, vmax=1)
            ax.set_title(titles_pt[h], fontsize=14, fontweight='bold', pad=10)
            ax.axis('off')
    
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
    cbar = fig.colorbar(im, cbar_ax, orientation='horizontal')
    cbar.set_label("Probabilidade Acumulada de Fogo (0.0 - 1.0)", fontsize=12, labelpad=5)
    
    # Updated Static Plot Title
    display_date = target_date.strftime('%d/%m/%Y')
    plt.suptitle(f"Evolução Temporal do Risco de Queimadas em MT - {display_date}", fontsize=16, fontweight='bold', y=0.98)
    plt.subplots_adjust(bottom=0.15, top=0.85, wspace=0.1)
    
    jpg_path = os.path.join(pred_dir, f"Painel_Evolutivo_Completo_{target_date.strftime('%Y_%m_%d')}.jpg")
    plt.savefig(jpg_path, dpi=1200, bbox_inches='tight', facecolor='white', format='jpg')
    plt.close(fig) 

    print("       -> Generating Interactive Web Map (HTML)...")
    cmap_html = cmap_base.copy()
    cmap_html.set_bad(alpha=0.0) 
    
    interactive_map = folium.Map(location=[-12.64, -55.42], zoom_start=6, tiles="CartoDB positron", control_scale=True)
    colormap = cm.LinearColormap(colors=['blue', 'yellow', 'red'], vmin=0.0, vmax=1.0)
    colormap.caption = 'Probabilidade Acumulada de Fogo'
    interactive_map.add_child(colormap)
    
    # Inject Custom Title Box onto Folium Map
    title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 650px; height: 35px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:16px; font-weight:bold; text-align:center; padding-top: 6px;
                    border-radius: 5px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
             Evolução Temporal do Risco de Queimadas em MT - {display_date}
        </div>
    '''
    interactive_map.get_root().html.add_child(folium.Element(title_html))
    
    for h in horizons:
        if h in pred_maps:
            masked_data = np.ma.masked_invalid(pred_maps[h])
            colorized = cmap_html(masked_data)
            img_data = np.uint8(colorized * 255)
            buffered = io.BytesIO()
            Image.fromarray(img_data).save(buffered, format="PNG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode()
            folium.raster_layers.ImageOverlay(image=f"data:image/png;base64,{img_b64}", bounds=map_bounds, opacity=0.65, name=f'Risco de Fogo: {titles_pt[h]}', show=(h == 1)).add_to(interactive_map)

    folium.GeoJson(mt_gdf, name="Limites Municipais", style_function=lambda x: {'color': '#333333', 'weight': 0.5, 'fillOpacity': 0}, tooltip=folium.GeoJsonTooltip(fields=[city_col], aliases=['Município:'], style="font-weight: bold; background: white;")).add_to(interactive_map)
    folium.GeoJson(fed_gdf, name="Rodovias Federais (BR)", style_function=lambda x: {'color': 'green', 'weight': 1}).add_to(interactive_map)
    folium.GeoJson(state_gdf, name="Rodovias Estaduais (MT)", style_function=lambda x: {'color': 'magenta', 'weight': 1}, show=False).add_to(interactive_map)
    folium.LayerControl(collapsed=False).add_to(interactive_map)
    
    html_path = os.path.join(pred_dir, f"Mapa_Interativo_Risco_{target_date.strftime('%Y_%m_%d')}.html")
    interactive_map.save(html_path)
    print(f"       [+] Saved outputs (TIFFs, JPG, HTML) to: {pred_dir}")


######################################################################
 STEP 5A: DYNAMIC DATE SELECTION & PER-VARIABLE IMPUTATION
######################################################################
[INFO] Target Prediction Date: 2026-09-10
[STATUS] Sequence Check: 0/14 required days available for 2026.
[WARNING] Missing 14 days. Initiating per-variable Welch T-Test historical matching...

  [VAR 0] Temperature
       -> No window data; using YTD distribution (165 days) as proxy.

  [VAR 1] VPD
       -> No window data; using YTD distribution (165 days) as proxy.

  [VAR 2] Soil Moisture
       -> No window data; using YTD distribution (165 days) as proxy.

  [VAR 3] CO
       -> No window data; using YTD distribution (165 days) as proxy.

  [VAR 4] EVI
       -> No window data; using YTD distribution (165 days) as proxy.

  Per-variable imputation sources
  Variable            Best Year    p-value
  ------------------------------------------
  Temperature              2022     0.81

/mnt/c/Users/daves/OneDrive/Pessoal/Notebooks/tf_env/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)



[SUCCESS] Final temporal-spatial tensor generated. Shape: torch.Size([1, 75, 128, 128])

######################################################################
 STEP 5B: EXECUTING CNN PREDICTION & MAPPING
######################################################################
 -> Executing forward pass for 1-day horizon...
 -> Executing forward pass for 7-day horizon...
 -> Executing forward pass for 14-day horizon...
 -> Executing forward pass for 30-day horizon...

[INFO] Assembling and saving visualizations...
       -> Generating ultra-high resolution JPG image (1x4 Grid) at 1200 DPI...
       -> Generating Interactive Web Map (HTML)...
       [+] Saved outputs (TIFFs, JPG, HTML) to: /home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Prediction


## Step 6: Memory Management
Deep learning inference on large spatial grids heavily consumes GPU VRAM and system RAM. This final cell flushes the memory, deletes the large input tensors, and clears the CUDA cache, ensuring the hardware is cleanly reset for subsequent runs.

In [4]:
# -----------------------------------------------------------------
# STEP 6: VRAM FLUSH
# -----------------------------------------------------------------
del x_input, current_year_data
gc.collect()
torch.cuda.empty_cache()
print("\n[SUCCESS] Notebook 2 Prediction Pipeline Complete. VRAM Flushed.")


[SUCCESS] Notebook 2 Prediction Pipeline Complete. VRAM Flushed.
